# spaCy
A single notebook covering core spaCy operations and a small corpus-analysis workflow on Shakespeare text.

## Setup
Install spaCy if needed and download the English model used in this notebook.

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

from collections import Counter
from pathlib import Path
from statistics import median

import spacy

nlp = spacy.load("en_core_web_sm")

## Basic Text Processing

In [ ]:
sample_text = "Hello! Welcome to the world of Natural Language Processing. Let's learn spaCy together."
doc = nlp(sample_text)
[token.text for token in doc]

In [ ]:
filtered_tokens = [token.text for token in doc if not token.is_stop]
filtered_tokens

In [ ]:
lemma_doc = nlp("running jumps easily programming programmers")
[token.lemma_ for token in lemma_doc]

In [ ]:
[(token.text, token.pos_, token.tag_) for token in doc]

In [ ]:
[(ent.text, ent.label_) for ent in doc.ents]

In [ ]:
sentence1 = nlp("I love apples.")
sentence2 = nlp("I enjoy eating fruit.")
sentence1.similarity(sentence2)

## Corpus Analysis
These cells expect a local `pg100.txt` file containing the Shakespeare corpus.

In [ ]:
corpus_path = Path("pg100.txt")
text = corpus_path.read_text(encoding="utf-8")
corpus_doc = nlp(text[:200000])

alpha_tokens = [token.text for token in corpus_doc if token.is_alpha]
sentences = list(corpus_doc.sents)

print(f"Word count in slice: {len(alpha_tokens)}")
print(f"Sentence count in slice: {len(sentences)}")

In [ ]:
word_lengths = [len(token) for token in alpha_tokens]
print({
    "min": min(word_lengths),
    "max": max(word_lengths),
    "avg": round(sum(word_lengths) / len(word_lengths), 2),
    "median": median(word_lengths),
})

In [ ]:
lemma_vocabulary = {token.lemma_.lower() for token in corpus_doc if token.is_alpha}
print(f"Unique lemma words in slice: {len(lemma_vocabulary)}")

In [ ]:
sentence_lengths = []
ranked_sentences = []
for sentence in sentences:
    words = [token.text for token in sentence if token.is_alpha]
    sentence_lengths.append(len(words))
    ranked_sentences.append((len(words), sentence.text.strip()))

print({
    "min": min(sentence_lengths),
    "max": max(sentence_lengths),
    "avg": round(sum(sentence_lengths) / len(sentence_lengths), 2),
    "median": median(sentence_lengths),
})
print("Shortest sentence:", min(ranked_sentences, key=lambda item: item[0])[1])
print("Longest sentence:", max(ranked_sentences, key=lambda item: item[0])[1])

In [ ]:
noun_counter = Counter(token.lemma_.lower() for token in corpus_doc if token.pos_ == "NOUN" and token.is_alpha)
noun_counter.most_common(10)